# Create Founder Level Education Dataset
This notebook cleans and aggregates the raw founder education dataset to the individual (`user_id`) level.

## Methodology and Design Decisions
1. **Degree Standardization**: Pre-classified `degree` is checked first. If empty, we standardize the degree raw string into **Bachelor**, **Master**, or **PhD** categories using a robust token-splitting keyword search.
2. **Discipline Indicators**: We identify specific fields of study using standardized `field` values and raw tokens in `field_raw`:
   * **STEM**: Engineering, Biology, Chemistry, Mathematics, Physics, Statistics, IT, Architecture.
   * **Business**: Business, Finance, Accounting, Marketing, Economics.
   * **Law**: Law degrees (JD, LLB, LLM, etc.).
   * **Medicine**: Medicine, Nursing, Pharmacy, Dentistry, healthcare fields.
   * **Specialized Degrees**: Separate indicators for **MBAs**, **PhDs**, **CS (Computer Science)**, **Economics**, and **Humanities**.
3. **Aggregation at Individual Level**: Group by `user_id` and aggregate:
   * Indicator variables: `max` (yields 1 if they have at least one degree in that category, 0 otherwise).
   * Years: `min` start year and `max` end year for each of the three degree categories (Bachelor, Master, PhD).


In [1]:
import csv
import os
import re
import time
import pandas as pd
import numpy as np


In [2]:
file_path = "../D - Data/D1 - Extracted Datasets/Founder_Education_List_[US-2000-2023].csv"
print("Loading dataset...")
df = pd.read_csv(file_path, low_memory=False)
print(f"Loaded {len(df):,} rows.")


Loading dataset...


Loaded 2,351,237 rows.


In [3]:
def classify_degree(row):
    deg = str(row['degree']).lower().strip()
    if deg == 'doctor':
        return 'PhD'
    elif deg in ['master', 'mba']:
        return 'Master'
    elif deg == 'bachelor':
        return 'Bachelor'
    
    deg_raw = str(row['degree_raw']).lower().strip()
    if not deg_raw or deg_raw == 'nan' or deg_raw == 'empty':
        return 'Other'
        
    raw_tokens = re.split(r'[\s\-\/]+', deg_raw)
    tokens = [re.sub(r'[^\w]', '', t) for t in raw_tokens if t]
    
    # Check PhD
    phd_keywords = {'phd', 'dphil', 'doctor', 'jd', 'md', 'dds', 'dmd', 'edd', 'psyd', 'sjd', 'dba'}
    if any(t in phd_keywords for t in tokens) or 'ph.d.' in deg_raw or 'doctor' in deg_raw or 'd.phil' in deg_raw:
        return 'PhD'
        
    # Check Master's
    master_keywords = {'master', 'masters', 'mba', 'ms', 'ma', 'msee', 'msme', 'mfa', 'mtech', 'me', 'msba', 'mpp', 'mpa', 'mph', 'meng', 'msw', 'msn', 'mls', 'med', 'mia', 'mha', 'mdes', 'mfin', 'msc'}
    if any(t in master_keywords for t in tokens) or 'master' in deg_raw or 'm.s.' in deg_raw or 'm.a.' in deg_raw or 'm.b.a.' in deg_raw or 'm.sc' in deg_raw:
        return 'Master'
        
    # Check Bachelor's
    bachelor_keywords = {'bachelor', 'bachelors', 'ba', 'bs', 'bba', 'bfa', 'btech', 'be', 'bse', 'bsw', 'ab', 'sb'}
    has_b_comb = False
    if len(tokens) >= 2:
        for idx in range(len(tokens) - 1):
            if tokens[idx] == 'b' and tokens[idx+1] in ['tech', 'e', 's', 'a', 'eng', 'sc', 'se', 'ba', 'bs']:
                has_b_comb = True
    
    if any(t in bachelor_keywords for t in tokens) or has_b_comb or 'bachelor' in deg_raw or 'b.s.' in deg_raw or 'b.a.' in deg_raw or 'b.b.a.' in deg_raw or 'b.sc' in deg_raw:
        return 'Bachelor'
        
    return 'Other'

def extract_year(val):
    if pd.isnull(val):
        return np.nan
    val_str = str(val).strip()
    if not val_str or val_str.lower() in ['empty', 'nan', 'none']:
        return np.nan
    first_4 = val_str[:4]
    if first_4.isdigit() and len(first_4) == 4:
        yr = int(first_4)
        if 1900 <= yr <= 2026:
            return yr
    match = re.search(r'\b(19\d{2}|20\d{2})\b', val_str)
    if match:
        return int(match.group(1))
    return np.nan

def classify_disciplines(row, classified_deg):
    field = str(row['field']).lower().strip()
    field_raw = str(row['field_raw']).lower().strip()
    degree_raw = str(row['degree_raw']).lower().strip()
    degree = str(row['degree']).lower().strip()
    
    # 1. STEM
    stem_fields = {'engineering', 'biology', 'chemistry', 'mathematics', 'physics', 'statistics', 'information technology', 'architecture'}
    is_stem = 0
    if field in stem_fields:
        is_stem = 1
    else:
        raw_tokens = re.split(r'[\s\-\/]+', field_raw)
        tokens = {re.sub(r'[^\w]', '', t) for t in raw_tokens if t}
        stem_keywords = {'engineering', 'biology', 'chemistry', 'mathematics', 'physics', 'statistics', 'math', 'stats', 'science', 'tech', 'technology', 'architecture', 'biomedical', 'biological', 'mechatronics', 'nanotechnology'}
        if any(t in stem_keywords for t in tokens) or 'computer science' in field_raw or 'information systems' in field_raw or 'software engineering' in field_raw:
            is_stem = 1
            
    # 2. Business
    business_fields = {'business', 'finance', 'accounting', 'marketing', 'economics'}
    is_business = 0
    if field in business_fields:
        is_business = 1
    else:
        raw_tokens = re.split(r'[\s\-\/]+', field_raw)
        tokens = {re.sub(r'[^\w]', '', t) for t in raw_tokens if t}
        business_keywords = {'business', 'finance', 'accounting', 'marketing', 'economics', 'commerce', 'management', 'mba', 'administration', 'strategy', 'entrepreneurship', 'operations'}
        if any(t in business_keywords for t in tokens) or 'business administration' in field_raw or 'financial' in field_raw or 'marketing' in field_raw:
            is_business = 1
            
    # 3. Law
    is_law = 0
    if field == 'law':
        is_law = 1
    else:
        raw_tokens = re.split(r'[\s\-\/]+', field_raw)
        tokens = {re.sub(r'[^\w]', '', t) for t in raw_tokens if t}
        law_keywords = {'law', 'legal', 'juris', 'jd', 'llm', 'llb', 'patent'}
        if any(t in law_keywords for t in tokens) or 'jd' in tokens or 'juris doctor' in field_raw:
            is_law = 1
            
    # 4. Medicine
    med_fields = {'medicine', 'nursing'}
    is_med = 0
    if field in med_fields:
        is_med = 1
    else:
        raw_tokens = re.split(r'[\s\-\/]+', field_raw)
        tokens = {re.sub(r'[^\w]', '', t) for t in raw_tokens if t}
        med_keywords = {'medicine', 'nursing', 'medical', 'nurse', 'pharmacy', 'clinical', 'healthcare', 'health', 'dentistry', 'surgery', 'chiropractic'}
        if any(t in med_keywords for t in tokens) or 'doctor of medicine' in field_raw or 'registered nurse' in field_raw:
            is_med = 1
            
    # 5. MBA
    is_mba = 0
    if degree == 'mba':
        is_mba = 1
    else:
        raw_tokens_deg = re.split(r'[\s\-\/]+', degree_raw)
        tokens_deg = {re.sub(r'[^\w]', '', t) for t in raw_tokens_deg if t}
        raw_tokens_fld = re.split(r'[\s\-\/]+', field_raw)
        tokens_fld = {re.sub(r'[^\w]', '', t) for t in raw_tokens_fld if t}
        if 'mba' in tokens_deg or 'mba' in tokens_fld or 'master of business administration' in degree_raw or 'master of business administration' in field_raw:
            is_mba = 1
            
    # 6. PhD
    is_phd = 1 if classified_deg == 'PhD' else 0
            
    # 7. CS
    is_cs = 0
    raw_tokens = re.split(r'[\s\-\/]+', field_raw)
    tokens = {re.sub(r'[^\w]', '', t) for t in raw_tokens if t}
    if 'cs' in tokens or 'computer science' in field_raw or 'software engineering' in field_raw or 'artificial intelligence' in field_raw or 'machine learning' in field_raw or 'computer engineering' in field_raw:
        is_cs = 1
        
    # 8. Economics
    is_econ = 0
    if field == 'economics':
        is_econ = 1
    else:
        raw_tokens = re.split(r'[\s\-\/]+', field_raw)
        tokens = {re.sub(r'[^\w]', '', t) for t in raw_tokens if t}
        if 'economics' in tokens or 'econ' in tokens:
            is_econ = 1
            
    # 9. Humanities
    is_humanities = 0
    raw_tokens = re.split(r'[\s\-\/]+', field_raw)
    tokens = {re.sub(r'[^\w]', '', t) for t in raw_tokens if t}
    hum_keywords = {'communications', 'psychology', 'history', 'english', 'sociology', 'anthropology', 'philosophy', 'literature', 'political science', 'journalism', 'arts', 'music', 'fine arts', 'design'}
    if any(t in hum_keywords for t in tokens) or 'political science' in field_raw or 'international relations' in field_raw:
        is_humanities = 1
        
    return pd.Series([is_stem, is_business, is_law, is_med, is_mba, is_phd, is_cs, is_econ, is_humanities])


In [4]:
print("Classifying degrees, years, and disciplines (this may take a few seconds)...")
start_time = time.time()
df['classified_degree'] = df.apply(classify_degree, axis=1)
df['start_year'] = df['startdate'].apply(extract_year)
df['end_year'] = df['enddate'].apply(extract_year)

cols = ['is_stem', 'is_business', 'is_law', 'is_med', 'is_mba', 'is_phd', 'is_cs', 'is_econ', 'is_humanities']
df[cols] = df.apply(lambda r: classify_disciplines(r, r['classified_degree']), axis=1)
print(f"Done in {time.time() - start_time:.2f} seconds.")


Classifying degrees, years, and disciplines (this may take a few seconds)...


Done in 101.98 seconds.


In [5]:
print("Aggregating at the individual level...")
agg_start = time.time()

df['is_bachelor'] = (df['classified_degree'] == 'Bachelor').astype(int)
df['is_master'] = (df['classified_degree'] == 'Master').astype(int)
df['is_phd'] = (df['classified_degree'] == 'PhD').astype(int)

df['bachelor_start'] = np.where(df['is_bachelor'] == 1, df['start_year'], np.nan)
df['bachelor_end'] = np.where(df['is_bachelor'] == 1, df['end_year'], np.nan)

df['master_start'] = np.where(df['is_master'] == 1, df['start_year'], np.nan)
df['master_end'] = np.where(df['is_master'] == 1, df['end_year'], np.nan)

df['phd_start'] = np.where(df['is_phd'] == 1, df['start_year'], np.nan)
df['phd_end'] = np.where(df['is_phd'] == 1, df['end_year'], np.nan)

agg_df = df.groupby('user_id').agg(
    has_bachelor=('is_bachelor', 'max'),
    bachelor_start_year=('bachelor_start', 'min'),
    bachelor_end_year=('bachelor_end', 'max'),
    has_master=('is_master', 'max'),
    master_start_year=('master_start', 'min'),
    master_end_year=('master_end', 'max'),
    has_phd=('is_phd', 'max'),
    phd_start_year=('phd_start', 'min'),
    phd_end_year=('phd_end', 'max'),
    has_stem=('is_stem', 'max'),
    has_business=('is_business', 'max'),
    has_law=('is_law', 'max'),
    has_med=('is_med', 'max'),
    has_mba=('is_mba', 'max'),
    has_cs=('is_cs', 'max'),
    has_econ=('is_econ', 'max'),
    has_humanities=('is_humanities', 'max')
).reset_index()

indicator_cols = ['has_bachelor', 'has_master', 'has_phd', 'has_stem', 'has_business', 'has_law', 'has_med', 'has_mba', 'has_cs', 'has_econ', 'has_humanities']
for col in indicator_cols:
    agg_df[col] = agg_df[col].fillna(0).astype(int)

print(f"Aggregated {len(agg_df):,} individuals in {time.time() - agg_start:.2f} seconds.")


Aggregating at the individual level...


Aggregated 987,658 individuals in 0.69 seconds.


In [6]:
print("Descriptive statistics of the aggregated individual education dataset:")
print(agg_df.describe())

out_dir = "../D - Data/D2 - Datasets for Matching"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "Founder_Education_Aggregated.csv")
print(f"Saving aggregated dataset to: {out_path}")
agg_df.to_csv(out_path, index=False)
print("Aggregation complete!")


Descriptive statistics of the aggregated individual education dataset:
            user_id   has_bachelor  bachelor_start_year  bachelor_end_year  \
count  9.876580e+05  987658.000000        602394.000000      601331.000000   
mean   5.155188e+08       0.749743          2002.345007        2006.209400   
std    4.897519e+08       0.433161            12.627895          12.531758   
min    1.000086e+06       0.000000          1900.000000        1900.000000   
25%    1.996808e+08       0.000000          1994.000000        1998.000000   
50%    4.099741e+08       1.000000          2004.000000        2008.000000   
75%    6.331860e+08       1.000000          2012.000000        2016.000000   
max    2.219679e+09       1.000000          2029.000000        2034.000000   

          has_master  master_start_year  master_end_year        has_phd  \
count  987658.000000      281724.000000    278873.000000  987658.000000   
mean        0.345937        2006.808114      2009.802753       0.097260   
s

Aggregation complete!
